In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent  # ← back to this

model = init_chat_model(
    "llama-3.3-70b-versatile",  # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,  # ← MUST be 0 for reliable tool calling
)

# create subagents

subagent_1 = create_agent(model=model, tools=[square_root])

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='e390c54b-d612-4ae1-a1d4-be396b9e1d8b'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3ayt74qch', 'function': {'arguments': '{"x":456}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 339, 'total_tokens': 356, 'completion_time': 0.034756353, 'completion_tokens_details': None, 'prompt_time': 0.070040789, 'prompt_tokens_details': None, 'queue_time': 0.153454891, 'total_time': 0.104797142}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3db0-d27e-7e32-bf3f-c98ced498e26-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '3ayt74qch', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata=

In [8]:
question = "What is the square of 56?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})